In [12]:
import pandas as pd

# Load the datasets
options_data = pd.read_csv('Option Code Information file for UNT - Final(Sheet1).csv')
claims_data = pd.read_csv('Claim Information file for UNT(Sheet1).csv')

# Standardize the truck identifier column for merging
options_data.rename(columns={"Truck": "Truck Number"}, inplace=True)

# Merge the data on 'Truck Number' with a left join to keep all trucks from options_data
merged_data = pd.merge(options_data, claims_data, on="Truck Number", how="left")

# Convert 'Scale Claim Cost' and 'Scale Labor Cost' to numeric, setting errors='coerce' to handle non-numeric entries
merged_data['Scale Claim Cost'] = pd.to_numeric(merged_data['Scale Claim Cost'], errors='coerce').fillna(0)
merged_data['Scale Labor Cost'] = pd.to_numeric(merged_data['Scale Labor Cost'], errors='coerce').fillna(0)

# Create a 'claim_flag' column: 1 if the truck is in claims_data, 0 otherwise
merged_data['claim_flag'] = merged_data['Scale Claim Cost'].apply(lambda x: 1 if x > 0 else 0)

# Display the first few rows to confirm the results
print(merged_data.head())

merged_data.to_csv('merged_data.csv', index=False)

# If you're running this in a notebook environment, provide a download link
from IPython.display import FileLink
FileLink('merged_data.csv')



  Truck Number    Style Attribute 1 Attribute 2 Attribute 3 Attribute 4  \
0      Truck 1  Style 1    Option 1   Option 83  Option 181  Option 204   
1      Truck 1  Style 1    Option 1   Option 83  Option 181  Option 204   
2      Truck 1  Style 1    Option 1   Option 83  Option 181  Option 204   
3      Truck 1  Style 1    Option 1   Option 83  Option 181  Option 204   
4      Truck 1  Style 1    Option 1   Option 83  Option 181  Option 204   

  Attribute 5 Attribute 6 Attribute 7 Attribute 8 Claim Number  \
0  Option 264  Option 281  Option 290  Option 294  Claim 13675   
1  Option 264  Option 281  Option 290  Option 294  Claim 18104   
2  Option 264  Option 281  Option 290  Option 294  Claim 20261   
3  Option 264  Option 281  Option 290  Option 294  Claim 21530   
4  Option 264  Option 281  Option 290  Option 294  Claim 21693   

   Scale Claim Cost  Scale Labor Cost  claim_flag  
0               0.0               0.0           0  
1               0.0               0.0           

/Users/lalith/merged_data.csv

In [8]:

pip install pulp


python(53573) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Obtaining dependency information for pulp from https://files.pythonhosted.org/packages/64/10/704c18b5960b3f9b10efcc859e11881ad90f1e44008e181d2b10cd305a63/PuLP-2.9.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 13.1 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
from pulp import LpMinimize, LpProblem, LpVariable, lpSum
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Define attributes and options per attribute (replace options_per_attribute with actual data count)
attributes = ["Attribute 1", "Attribute 2", "Attribute 3", "Attribute 4", "Attribute 5", "Attribute 6", "Attribute 7", "Attribute 8"]
options_per_attribute = 100  # assuming 10 options per attribute

# Generate a sample claim cost dictionary (replace with actual claim costs for accurate results)
claim_costs = {(attr, opt): np.random.randint(50, 500) for attr in attributes for opt in range(options_per_attribute)}

# Initialize the optimization model
model = LpProblem("Truck_Configuration_Optimization", LpMinimize)

# Define decision variables
decision_vars = {(attr, opt): LpVariable(f"{attr}_{opt}", cat="Binary") for attr in attributes for opt in range(options_per_attribute)}

# Objective Function: Minimize total claim cost
model += lpSum(claim_costs[attr, opt] * decision_vars[attr, opt] for attr in attributes for opt in range(options_per_attribute)), "Total Claim Cost"

# Constraints: Only one option per attribute should be selected
for attr in attributes:
    model += lpSum(decision_vars[attr, opt] for opt in range(options_per_attribute)) == 1, f"One_option_per_{attr}"

# Solve the optimization problem
model.solve()

# Output the optimal selection and total claim cost
optimal_selection = {attr: opt for (attr, opt), var in decision_vars.items() if var.varValue == 1}
total_claim_cost = model.objective.value()

print("Optimal Selection per Attribute:", optimal_selection)
print("Total Claim Cost:", total_claim_cost)


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/lalith/anaconda3/lib/python3.11/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/fz/5pvnz1r94rd7f5z3skq5mbxr0000gn/T/8ace6975e1a943ea8f283ed98768d3ea-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/fz/5pvnz1r94rd7f5z3skq5mbxr0000gn/T/8ace6975e1a943ea8f283ed98768d3ea-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 13 COLUMNS
At line 3214 RHS
At line 3223 BOUNDS
At line 4024 ENDATA
Problem MODEL has 8 rows, 800 columns and 800 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 413 - 0.00 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from 413 to -1.79769e+308
Probing was tried 0 times and created 0 cuts of which 0 w

python(53887) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [2]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load datasets
option_code_info_df = pd.read_csv('Option Code Information file for UNT - Final(Sheet1).csv')
claim_info_df = pd.read_csv('Claim Information file for UNT(Sheet1).csv')

# Rename and merge with a left join to retain all trucks
option_code_info_df.rename(columns={'Truck': 'Truck Number'}, inplace=True)
merged_df = pd.merge(option_code_info_df, claim_info_df, on='Truck Number', how='left')

# Ordinal Encoding for 'Scale Claim Cost' and 'Scale Labor Cost'
ordinal_mapping = {'Very Low': 1, 'Low': 2, 'Medium': 3, 'High': 4, 'Very High': 5}
merged_df['Scale Claim Cost'] = merged_df['Scale Claim Cost'].map(ordinal_mapping)
merged_df['Scale Labor Cost'] = merged_df['Scale Labor Cost'].map(ordinal_mapping)

# Fill NaN values for claim-related columns (optional: could fill with 0 or another value depending on the use case)
merged_df['Scale Claim Cost'].fillna(0, inplace=True)  # Assuming 0 represents no claims
merged_df['Scale Labor Cost'].fillna(0, inplace=True)

# Select features and target variables
X = merged_df.drop(columns=['Truck Number', 'Claim Number', 'Scale Claim Cost'])
y = merged_df['Scale Claim Cost']

# One-Hot Encode categorical variables
encoder = OneHotEncoder(sparse=False, drop='first')
X_encoded = encoder.fit_transform(X)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)

# Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

# Logistic Regression Metrics
log_reg_accuracy = accuracy_score(y_test, y_pred_log_reg)
log_reg_report = classification_report(y_test, y_pred_log_reg)

# Naive Bayes Model
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Naive Bayes Metrics
nb_accuracy = accuracy_score(y_test, y_pred_nb)
nb_report = classification_report(y_test, y_pred_nb)

# Print results
print("Logistic Regression Accuracy:", log_reg_accuracy)
print("Logistic Regression Report:\n", log_reg_report)
print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes Report:\n", nb_report)


/Users/lalith/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Logistic Regression Accuracy: 0.8429634625111395
Logistic Regression Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      4577
         1.0       0.56      0.33      0.41      4513
         2.0       0.85      0.96      0.90     29877
         3.0       0.52      0.04      0.07      1933
         4.0       0.49      0.24      0.32       485
         5.0       0.60      0.18      0.28       134

    accuracy                           0.84     41519
   macro avg       0.67      0.46      0.50     41519
weighted avg       0.81      0.84      0.81     41519

Naive Bayes Accuracy: 0.7695031190539271
Naive Bayes Report:
               precision    recall  f1-score   support

         0.0       0.99      0.45      0.62      4577
         1.0       0.48      0.48      0.48      4513
         2.0       0.80      0.92      0.86     29877
         3.0       0.43      0.04      0.08      1933
         4.0       0.46      0.20      0.28   

In [3]:
from sklearn.ensemble import RandomForestClassifier

# Instantiate and train the Random Forest model
# Using 'Scale Claim Cost' as the target variable for warranty cost analysis
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

# Get feature importance from the trained model
feature_importances = rf_model.feature_importances_

# Match feature importances with the feature names
feature_names = encoder.get_feature_names_out(X.columns)
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Display the top features affecting warranty costs
feature_importance_df.head(20)


,Feature,Importance
323,Scale Labor Cost_2.0,0.423280
322,Scale Labor Cost_1.0,0.246601
324,Scale Labor Cost_3.0,0.026460
196,Attribute 3_Option 185,0.008394
274,Attribute 5_Option 265,0.008010
217,Attribute 4_Option 207,0.007341
194,Attribute 3_Option 183,0.006771
291,Attribute 6_Option 284,0.006009
306,Attribute 8_Option 300,0.005748
297,Attribute 6_Option283,0.005448


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define the parameter grid for Randomized Search
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Instantiate the Random Forest classifier
rf_model = RandomForestClassifier(random_state=42)

# Instantiate the RandomizedSearchCV with cross-validation
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=50,  # Number of random combinations to try
    cv=5,       # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Fit the model on the training data
random_search.fit(X_train, y_train)

# Best hyperparameters found
best_params = random_search.best_params_
print("Best Parameters:", best_params)

# Best model
best_rf_model = random_search.best_estimator_

# Evaluate on the test set
y_pred = best_rf_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
